# GDELT Data Profiling and Description Generation

This notebook automates profiling tables in your GDELT BigQuery dataset using Google Cloud Dataplex, then uses the Dataplex Data Documentation API (Gemini-backed) to generate and apply table and column descriptions.

## When to run

- Run **after** `01_gdelt_data_prep.ipynb` so your dataset has tables to scan.
- Run **after** `02_gdelt_graph_create.ipynb` if you want descriptions for graph-related tables as well as raw GDELT copies (optional).

## GCP setup

- Enable the **Dataplex API** (and related BigQuery access) for your project.
- Install dependencies from the repo root (`pip install -r requirements.txt`); **`google-cloud-dataplex`** is listed there.

## Steps

1. **Environment setup**: Load configuration and initialize clients.
2. **Dataplex data profiling**: Run profile scans per table.
3. **Documentation generation**: Call the Dataplex Data Documentation API for descriptions.
4. **Apply to BigQuery**: Write descriptions into BigQuery table/column metadata.


In [ ]:
# Import required libraries
import os
import time
from google.cloud import bigquery
from google.cloud import dataplex_v1
from google.api_core.exceptions import AlreadyExists, InvalidArgument, ResourceExhausted
from google.cloud.exceptions import NotFound

# Import configuration from config.py
from config import (
    GCP_PROJECT_ID,
    PROJECT_REGION,
    BIGQUERY_DATASET,
    print_config,
    validate_config
)

# Display configuration
print_config()

# Validate configuration
if not validate_config():
    print("\n⚠️  Please update config.py before proceeding!")
else:
    print("\n✅ Configuration validated successfully!")

# Initialize clients
bq_client = bigquery.Client(project=GCP_PROJECT_ID)
dataplex_client = dataplex_v1.DataScanServiceClient()

# Get all tables in the dataset
dataset_ref = f"{GCP_PROJECT_ID}.{BIGQUERY_DATASET}"
tables = list(bq_client.list_tables(dataset_ref))
_all_ids = [table.table_id for table in tables]
_skipped_staging = sorted(t for t in _all_ids if t.startswith("staging_"))
table_names = [t for t in _all_ids if not t.startswith("staging_")]
print(f"\n📊 Tables in dataset: {len(_all_ids)}; profiling/documentation: {len(table_names)}")
if _skipped_staging:
    print(f"   ⏭️  Excluding ephemeral staging_* tables: {', '.join(_skipped_staging)}")
print(f"   Candidates: {', '.join(table_names)}")


def _call_dataplex_with_backoff(fn, op_label="operation"):
    """Retry Dataplex calls when quota returns 429 (often ~30 req/min/user/region)."""
    delay = float(os.environ.get("DATAPLEX_429_BACKOFF_SEC", "10"))
    max_attempts = int(os.environ.get("DATAPLEX_429_MAX_ATTEMPTS", "24"))
    for attempt in range(max_attempts):
        try:
            return fn()
        except ResourceExhausted:
            if attempt == max_attempts - 1:
                raise
            print(f"   ⚠️  Dataplex rate limit (429) during {op_label}; waiting {delay:.0f}s...")
            time.sleep(delay)
            delay = min(delay * 1.5, 120.0)

## 1. Dataplex Data Profiling

First, we create and run a Dataplex Data Profile scan for each table. This gathers statistics about the data (null counts, distributions, etc.) which grounds the subsequent Gemini generation.

In [ ]:
def run_data_profile_scan(project_id, location, dataset_id, table_id):
    """Create and start a Dataplex Data Profile scan for a BigQuery table."""
    print(f"\n🔍 Processing profile scan for {table_id}...")

    fqtn = f"{project_id}.{dataset_id}.{table_id}"
    try:
        bq_client.get_table(fqtn)
    except NotFound:
        print("   ⏭️  BigQuery table not found — skipping profile scan.")
        return None

    # Create a unique scan ID
    # Stable ID so re-running this cell reuses the scan (AlreadyExists) instead of creating duplicates
    scan_id = (
        f"profile-{dataset_id}-{table_id}".replace("_", "-").lower()[:63].rstrip("-")
    )
    
    # Define the resource name
    resource_name = f"//bigquery.googleapis.com/projects/{project_id}/datasets/{dataset_id}/tables/{table_id}"
    
    # Create the DataScan object
    data_scan = dataplex_v1.DataScan(
        data=dataplex_v1.DataSource(resource=resource_name),
        data_profile_spec=dataplex_v1.DataProfileSpec(),
        execution_spec=dataplex_v1.DataScan.ExecutionSpec(
            trigger=dataplex_v1.Trigger(on_demand=dataplex_v1.Trigger.OnDemand())
        )
    )
    
    # Create the scan
    parent = f"projects/{project_id}/locations/{location}"
    request = dataplex_v1.CreateDataScanRequest(
        parent=parent,
        data_scan_id=scan_id,
        data_scan=data_scan
    )
    
    try:
        print(f"   Creating scan {scan_id}...")
        operation = _call_dataplex_with_backoff(
            lambda: dataplex_client.create_data_scan(request=request),
            "create_data_scan",
        )
        scan = _call_dataplex_with_backoff(
            lambda: operation.result(),
            "create_data_scan_wait",
        )
        print(f"   Scan created successfully.")
    except AlreadyExists:
        print(f"   Scan already exists.")
        scan = _call_dataplex_with_backoff(
            lambda: dataplex_client.get_data_scan(name=f"{parent}/dataScans/{scan_id}"),
            "get_data_scan",
        )
    except InvalidArgument as e:
        if "not found" in str(e).lower():
            print("   ⏭️  Dataplex: BigQuery source not available — skipping profile scan.")
            return None
        raise

    # Run the scan
    print(f"   Starting scan...")
    run_request = dataplex_v1.RunDataScanRequest(name=scan.name)
    try:
        run_response = _call_dataplex_with_backoff(
            lambda: dataplex_client.run_data_scan(request=run_request),
            "run_data_scan",
        )
    except InvalidArgument as e:
        if "not found" in str(e).lower():
            print("   ⏭️  Dataplex run: BigQuery source not available — skipping.")
            return None
        raise
    job_name = run_response.job.name
    print(f"   Scan started: {job_name.split('/')[-1]}")

    return job_name

# Start all profile scans — jobs run in the background in Dataplex
# No need to wait; proceed to the next step when ready
# Pause slightly between tables to stay under Dataplex per-minute quotas (tunable via env).
_table_pause = float(os.environ.get("DATAPLEX_TABLE_PAUSE_SEC", "3"))
profile_jobs = {}
for i, table_name in enumerate(table_names):
    job_name = run_data_profile_scan(GCP_PROJECT_ID, PROJECT_REGION, BIGQUERY_DATASET, table_name)
    if job_name is not None:
        profile_jobs[table_name] = job_name
    if i < len(table_names) - 1:
        time.sleep(_table_pause)

print("\n✅ All profile scans started. Jobs are running in Dataplex.")
print("\n📋 Job summary:")
for table, job in profile_jobs.items():
    print(f"   {table}: {job.split('/')[-1]}")
print("\nYou can monitor progress in the Dataplex console or run the next cell when scans have completed.")

## 2. Gemini Description Generation & BigQuery Update

Next, we use the Dataplex Data Documentation API to generate descriptions using Gemini. This API returns the AI-generated table overview and column descriptions, which we then apply directly to the BigQuery schema.

In [ ]:
import subprocess

from google.api_core.exceptions import FailedPrecondition
from google.cloud.dataplex_v1.types import GetDataScanJobRequest

_GEMINI_CLOUD_API = "cloudaicompanion.googleapis.com"


def _maybe_enable_gemini_cloud_api():
    """Dataplex Data Documentation requires Gemini for Google Cloud API on the project."""
    if os.environ.get("DATAPLEX_AUTO_ENABLE_GEMINI_API", "1").strip().lower() in (
        "0",
        "false",
        "no",
    ):
        return
    print(f"Ensuring {_GEMINI_CLOUD_API} is enabled for {GCP_PROJECT_ID}...")
    proc = subprocess.run(
        [
            "gcloud",
            "services",
            "enable",
            _GEMINI_CLOUD_API,
            f"--project={GCP_PROJECT_ID}",
        ],
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        print(proc.stderr.strip() or proc.stdout.strip() or f"gcloud exited {proc.returncode}")
    else:
        print("   gcloud services enable completed (already enabled or newly enabled).")
    wait_s = int(os.environ.get("DATAPLEX_GEMINI_API_PROPAGATION_SEC", "15"))
    if wait_s > 0:
        print(f"   Waiting {wait_s}s for API activation to propagate...")
        time.sleep(wait_s)


def start_documentation_scan(project_id, location, dataset_id, table_id):
    """Start a Dataplex Data Documentation scan for a BigQuery table."""
    print(f"\n📝 Starting documentation scan for {table_id}...")

    fqtn = f"{project_id}.{dataset_id}.{table_id}"
    try:
        bq_client.get_table(fqtn)
    except NotFound:
        print("   ⏭️  BigQuery table not found — skipping documentation scan.")
        return None

    # Create a unique scan ID
    scan_id = (
        f"docs-{dataset_id}-{table_id}".replace("_", "-").lower()[:63].rstrip("-")
    )
    
    # Define the resource name
    resource_name = f"//bigquery.googleapis.com/projects/{project_id}/datasets/{dataset_id}/tables/{table_id}"
    
    # Create the DataScan object for Data Documentation
    data_scan = dataplex_v1.DataScan(
        data=dataplex_v1.DataSource(resource=resource_name),
        data_documentation_spec=dataplex_v1.DataDocumentationSpec(),
        type_=dataplex_v1.DataScanType.DATA_DOCUMENTATION,
        execution_spec=dataplex_v1.DataScan.ExecutionSpec(
            trigger=dataplex_v1.Trigger(on_demand=dataplex_v1.Trigger.OnDemand())
        )
    )
    
    # Create the scan
    parent = f"projects/{project_id}/locations/{location}"
    request = dataplex_v1.CreateDataScanRequest(
        parent=parent,
        data_scan_id=scan_id,
        data_scan=data_scan
    )
    
    try:
        print(f"   Creating documentation scan {scan_id}...")
        operation = _call_dataplex_with_backoff(
            lambda: dataplex_client.create_data_scan(request=request),
            "create_documentation_scan",
        )
        scan = _call_dataplex_with_backoff(
            lambda: operation.result(),
            "create_documentation_scan_wait",
        )
        print(f"   Scan created successfully.")
    except AlreadyExists:
        print(f"   Scan already exists.")
        scan = _call_dataplex_with_backoff(
            lambda: dataplex_client.get_data_scan(name=f"{parent}/dataScans/{scan_id}"),
            "get_data_scan",
        )
    except InvalidArgument as e:
        if "not found" in str(e).lower():
            print("   ⏭️  Dataplex: BigQuery source not available — skipping documentation scan.")
            return None
        raise

    # Run the scan
    print(f"   Starting scan...")
    run_request = dataplex_v1.RunDataScanRequest(name=scan.name)
    try:
        run_response = _call_dataplex_with_backoff(
            lambda: dataplex_client.run_data_scan(request=run_request),
            "run_data_scan",
        )
    except InvalidArgument as e:
        if "not found" in str(e).lower():
            print("   ⏭️  Dataplex run: BigQuery source not available — skipping.")
            return None
        raise
    job_name = run_response.job.name
    print(f"   Scan started: {job_name.split('/')[-1]}")

    return job_name

def apply_descriptions_to_bq(project_id, dataset_id, table_id, job_result):
    """Extract descriptions from job result and apply to BigQuery."""
    if not job_result or not job_result.data_documentation_result:
        print(f"   ❌ No documentation results found for {table_id}.")
        return False
        
    doc_result = job_result.data_documentation_result
    table_overview = doc_result.table_result.overview
    
    print(f"\nApplying descriptions for {table_id}...")
    print(f"   Generated Table Description: {table_overview[:100]}...")
    
    # Map column descriptions
    col_descriptions = {}
    for field in doc_result.table_result.schema.fields:
        if field.description:
            col_descriptions[field.name] = field.description
            
    print(f"   Generated descriptions for {len(col_descriptions)} columns.")
    
    # Apply to BigQuery
    table_ref = f"{project_id}.{dataset_id}.{table_id}"
    bq_table = bq_client.get_table(table_ref)
    
    # Update table description
    bq_table.description = table_overview
    
    # Update column descriptions
    new_schema = []
    for field in bq_table.schema:
        new_desc = col_descriptions.get(field.name, field.description)
        new_field = bigquery.SchemaField(
            name=field.name,
            field_type=field.field_type,
            mode=field.mode,
            description=new_desc,
            fields=field.fields,
            policy_tags=field.policy_tags,
            precision=field.precision,
            scale=field.scale,
            max_length=field.max_length,
        )
        new_schema.append(new_field)
        
    bq_table.schema = new_schema
    
    # Save changes
    bq_client.update_table(bq_table, ["description", "schema"])
    print(f"   ✅ Successfully updated BigQuery schema for {table_id}!")
    return True

_skip_doc = os.environ.get("SKIP_DATAPLEX_DOCUMENTATION", "").strip().lower() in (
    "1",
    "true",
    "yes",
)

if _skip_doc:
    print(
        "\n⏭️  SKIP_DATAPLEX_DOCUMENTATION is set — skipping Gemini-backed documentation scans "
        "(section 1 profiling is unchanged)."
    )
else:
    _maybe_enable_gemini_cloud_api()

    _table_pause = float(os.environ.get("DATAPLEX_TABLE_PAUSE_SEC", "3"))
    doc_jobs = {}
    try:
        for i, table_name in enumerate(table_names):
            job_name = start_documentation_scan(
                GCP_PROJECT_ID, PROJECT_REGION, BIGQUERY_DATASET, table_name
            )
            if job_name is not None:
                doc_jobs[table_name] = job_name
            if i < len(table_names) - 1:
                time.sleep(_table_pause)
    except FailedPrecondition as exc:
        msg = str(exc).lower()
        if "cloudaicompanion" in msg or "gemini for google cloud" in msg:
            print(
                "\n⚠️  Dataplex documentation needs Gemini for Google Cloud API "
                f"({_GEMINI_CLOUD_API})."
            )
            print(
                "   Enable it for this project (wait a few minutes after first enable), then re-run this cell:\n"
                f"   gcloud services enable {_GEMINI_CLOUD_API} --project={GCP_PROJECT_ID}"
            )
            print(
                "   Or set SKIP_DATAPLEX_DOCUMENTATION=1 to skip AI descriptions "
                "when running the full notebook unattended."
            )
            doc_jobs = {}
        else:
            raise

    if doc_jobs:
        # Poll for completion and apply descriptions (view=FULL includes result payloads).
        print("\n⏳ Waiting for all documentation scans to complete...")
        pending_jobs = list(doc_jobs.keys())
        while pending_jobs:
            for table_name in pending_jobs[:]:
                job_name = doc_jobs[table_name]
                job = _call_dataplex_with_backoff(
                    lambda: dataplex_client.get_data_scan_job(
                        request=GetDataScanJobRequest(
                            name=job_name,
                            view=GetDataScanJobRequest.DataScanJobView.FULL,
                        )
                    ),
                    "get_data_scan_job",
                )

                if job.state == dataplex_v1.DataScanJob.State.SUCCEEDED:
                    print(f"   ✅ Job {job_name.split('/')[-1]} succeeded!")
                    apply_descriptions_to_bq(GCP_PROJECT_ID, BIGQUERY_DATASET, table_name, job)
                    pending_jobs.remove(table_name)

                elif job.state in [
                    dataplex_v1.DataScanJob.State.FAILED,
                    dataplex_v1.DataScanJob.State.CANCELLED,
                ]:
                    print(
                        f"   ❌ Job {job_name.split('/')[-1]} failed or was cancelled: {job.message}"
                    )
                    pending_jobs.remove(table_name)

            if pending_jobs:
                print(f"   {len(pending_jobs)} jobs still running. Waiting 15 seconds...")
                time.sleep(15)

        print("\n🎉 All table descriptions generated and applied successfully!")
    elif not _skip_doc:
        print("\n⚠️  No documentation jobs ran — BigQuery descriptions unchanged.")